In [ ]:
import json
import re
import numpy as np
import pandas as pd
import torch

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
apartments_df = pd.read_csv('apartments_data-1.csv')

### Loading in Qwen Model  

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {"dtype": torch.float16}
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"dtype": torch.float32}

print(f"Loading {model_name} on {device}...")
generator = pipeline(
    "text-generation",
    model=model_name,
    device=device,
    **model_kwargs
)
print("Model loaded.")


### Adding in Hard Filter Schema

In [ ]:
BINARY_FILTER_COLUMNS = [
    "in_unit_laundry",
    "dishwasher",
    "central_air",
    "parking_included",
    "gym_in_building",
    "balcony",
    "pets_allowed",
    "heat_included",
    "water_included"
]

NUMERIC_FILTER_COLUMNS = [
    "rent_max",
    "bedrooms_min",
    "bathrooms_min",
    "sqft_min"
]

OTHER_FILTER_COLUMNS = [
    "neighborhood"
]

### Parsing and Normalization

In [ ]:
def extract_json_from_text(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))


def normalize_binary(value):
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        return 1 if value >= 1 else 0
    if isinstance(value, str):
        return 1 if value.strip().lower() in {"1", "true", "yes"} else 0
    return 0


def normalize_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        cleaned = re.sub(r"[^\d.]", "", value)
        if cleaned == "":
            return None
        return float(cleaned)
    return None


def normalize_neighborhood(value):
    if value is None:
        return []

    if isinstance(value, str):
        val = value.strip()
        return [val] if val else []

    if isinstance(value, list):
        cleaned = []
        for v in value:
            if isinstance(v, str) and v.strip():
                cleaned.append(v.strip())
        return cleaned

    return []


def normalize_filter_output(raw_output):
    normalized = {}

    for key in BINARY_FILTER_COLUMNS:
        normalized[key] = normalize_binary(raw_output.get(key, 0))

    normalized["rent_max"] = normalize_number(raw_output.get("rent_max"))
    normalized["bedrooms_min"] = normalize_number(raw_output.get("bedrooms_min"))
    normalized["bathrooms_min"] = normalize_number(raw_output.get("bathrooms_min"))
    normalized["sqft_min"] = normalize_number(raw_output.get("sqft_min"))

    normalized["neighborhood"] = normalize_neighborhood(raw_output.get("neighborhood"))

    return normalized

### LLM Extraction



In [ ]:
def extract_filters_llm(user_text, generator, max_new_tokens=300):
    prompt = f"""
You are an information extraction system for apartment search.

Read the user's apartment description and return ONLY a valid JSON object with exactly these keys:

- in_unit_laundry
- dishwasher
- central_air
- parking_included
- gym_in_building
- balcony
- pets_allowed
- heat_included
- water_included
- neighborhood
- rent_max
- bedrooms_min
- bathrooms_min
- sqft_min

Rules:
- For the amenity keys, return 1 if clearly requested, otherwise 0.
- neighborhood should be a JSON list of desired neighborhoods. If none are mentioned, return [].
- rent_max should be the maximum rent the user wants to pay. If not mentioned, return null.
- bedrooms_min should be the minimum number of bedrooms requested. If not mentioned, return null.
- bathrooms_min should be the minimum number of bathrooms requested. If not mentioned, return null.
- sqft_min should be the minimum square footage requested. If not mentioned, return null.
- Do not include explanations.
- Output JSON only.

User description:
\"\"\"{user_text}\"\"\"
"""

    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    raw_text = response[0]["generated_text"].strip()
    parsed = extract_json_from_text(raw_text)
    return normalize_filter_output(parsed)


def safe_extract_filters_llm(user_text, generator):
    try:
        return extract_filters_llm(user_text, generator)
    except Exception:
        fallback = {key: 0 for key in BINARY_FILTER_COLUMNS}
        fallback.update({
            "neighborhood": [],
            "rent_max": None,
            "bedrooms_min": None,
            "bathrooms_min": None,
            "sqft_min": None
        })
        return fallback

### Apply Filters

In [ ]:
def apply_all_filters(apartments_df, filters_dict):
    df = apartments_df.copy()

    # binary amenity filters
    for col in BINARY_FILTER_COLUMNS:
        if filters_dict[col] == 1:
            df = df[df[col] == 1]

    # neighborhood filter
    desired_neighborhoods = filters_dict["neighborhood"]
    if desired_neighborhoods:
        desired_lower = {n.lower() for n in desired_neighborhoods}
        df = df[df["neighborhood"].astype(str).str.lower().isin(desired_lower)]

    # numeric filters
    if filters_dict["rent_max"] is not None:
        df = df[df["rent"] <= filters_dict["rent_max"]]

    if filters_dict["bedrooms_min"] is not None:
        df = df[df["bedrooms"] >= filters_dict["bedrooms_min"]]

    if filters_dict["bathrooms_min"] is not None:
        df = df[df["bathrooms"] >= filters_dict["bathrooms_min"]]

    if filters_dict["sqft_min"] is not None:
        df = df[df["sqft"] >= filters_dict["sqft_min"]]

    return df

### Text Prep

In [ ]:
def combine_listing_text(df):
    df = df.copy()

    text_cols = ["listing_description", "review_1", "review_2", "review_3"]
    for col in text_cols:
        if col not in df.columns:
            df[col] = ""

    df[text_cols] = df[text_cols].fillna("").astype(str)

    df["combined_text"] = (
        df["listing_description"] + " " +
        df["listing_description"] + " " +
        df["review_1"] + " " +
        df["review_2"] + " " +
        df["review_3"]
    ).str.strip()

    return df


def min_max_scale(series):
    series = series.astype(float)
    if series.max() == series.min():
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / (series.max() - series.min())

### Second Pass Rankings

In [ ]:
def rank_apartments_second_pass(
    filtered_df,
    user_text,
    top_n=10,
    similarity_threshold=0.08,
    similarity_weight=0.85,
    ctr_weight=0.15
):
    if filtered_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = combine_listing_text(filtered_df)

    corpus = df["combined_text"].tolist() + [user_text]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )

    tfidf_matrix = vectorizer.fit_transform(corpus)
    apt_matrix = tfidf_matrix[:-1]
    user_vector = tfidf_matrix[-1]

    df["text_similarity"] = cosine_similarity(apt_matrix, user_vector).flatten()
    df["click_through_rate"] = df["click_through_rate"].fillna(0).astype(float)
    df["ctr_norm"] = min_max_scale(df["click_through_rate"])

    df["final_score"] = (
        similarity_weight * df["text_similarity"] +
        ctr_weight * df["ctr_norm"]
    )

    strong_matches = df[df["text_similarity"] >= similarity_threshold].copy()
    strong_matches = strong_matches.sort_values("final_score", ascending=False)

    if len(strong_matches) >= top_n:
        recommended = strong_matches.head(top_n).copy()
    else:
        recommended_ids = strong_matches["listing_id"].tolist()
        fillers = df[~df["listing_id"].isin(recommended_ids)].copy()
        fillers = fillers.sort_values("final_score", ascending=False)
        recommended = pd.concat(
            [strong_matches, fillers.head(top_n - len(strong_matches))],
            ignore_index=True
        )

    other_apartments = df[~df["listing_id"].isin(recommended["listing_id"])].copy()
    other_apartments = other_apartments.sort_values(
        ["text_similarity", "click_through_rate"],
        ascending=False
    )

    return recommended, other_apartments

### Full Pipeline

In [ ]:
def recommend_apartments_llm(apartments_df, user_description, generator, top_n=10):
    extracted_filters = safe_extract_filters_llm(user_description, generator)
    filtered_df = apply_all_filters(apartments_df, extracted_filters)

    top_recs, other_apartments = rank_apartments_second_pass(
        filtered_df=filtered_df,
        user_text=user_description,
        top_n=top_n
    )

    return {
        "extracted_filters": extracted_filters,
        "num_after_first_pass": len(filtered_df),
        "top_10_recommendations": top_recs,
        "other_apartments": other_apartments
    }

### Example Usage

In [ ]:
user_description_1 = """
I want a one bedroom apartment in Back Bay or South End for no more than $3200.
I want at least one bathroom and at least 700 square feet.
It must have in-unit laundry, dishwasher, and central air.
A gym in the building would be great too.
"""

user_description_2 = """
I want a one bedroom apartment in any neighborhood for no more than $2000.
I want one bathroom and at least 200 square feet.
It must have dishwasher and central air.
A gym in the building would be great too. It would be great if the apartment has good reviews, and is in a young and hip part of town.
"""

results = recommend_apartments_llm(apartments_df, user_description_2, generator, top_n=10)

print("Extracted filters:")
print(results["extracted_filters"])

print("\nListings remaining after first pass:")
print(results["num_after_first_pass"])

cols_to_show = [
    "listing_id",
    "neighborhood",
    "rent",
    "bedrooms",
    "bathrooms",
    "sqft",
    "click_through_rate",
    "text_similarity",
    "final_score"
]

print("\nTop 10 recommendations:")
display(results["top_10_recommendations"][cols_to_show])

print("\nOther apartments:")
display(results["other_apartments"][cols_to_show].head(20))